# PentestingAgent 

This version trains on a curriculum of randomized networks instead of repeatedly fitting one static simulation.



## 1. Install dependencies

In [ ]:
!pip install nasim -q
!pip install torch-geometric -q
!pip install scikit-learn -q

In [ ]:
import numpy, torch, torch_geometric, nasim;
print(numpy.__version__, torch.__version__, torch_geometric.__version__)

## 2. Imports

In [ ]:
import nasim
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data
from enum import Enum
from pathlib import Path
import yaml
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from nasim.envs.action import SubnetScan, ServiceScan, OSScan, Exploit, PrivilegeEscalation, ProcessScan
from nasim.envs.host_vector import HostVector
from nasim.envs.utils import AccessLevel

GLOBAL_SEED = 2026
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)


## 3. Generate train/validation/test scenario banks

Every scenario uses the same semantic vocabulary (`linux/windows`, three services, and two processes) but varies host count, topology, host configuration, firewall rules, action costs/probabilities, and honeypot location. The split seeds never overlap.


In [ ]:
SCENARIO_DIR = Path("scenario_bank")
SCENARIO_DIR.mkdir(exist_ok=True)

OS_VOCAB = ["linux", "windows"]
SERVICE_VOCAB = ["ssh", "ftp", "http"]
PROCESS_VOCAB = ["tomcat", "daclsvc"]
MAX_SUBNET_ID = 4
MAX_HOST_ID = 4  # subnet 3 contains between 2 and 5 hosts

TOPOLOGY_TEMPLATES = [
    # Internet -> DMZ; DMZ branches to sensitive/user; user -> final sensitive.
    {
        "edges": [(0, 1), (1, 2), (1, 3), (2, 3), (3, 4)],
        "parents": {1: 0, 2: 1, 3: 1, 4: 3},
    },
    # Deeper chain.
    {
        "edges": [(0, 1), (1, 2), (2, 3), (3, 4)],
        "parents": {1: 0, 2: 1, 3: 2, 4: 3},
    },
    # User subnet is the central pivot.
    {
        "edges": [(0, 1), (1, 3), (3, 2), (3, 4)],
        "parents": {1: 0, 3: 1, 2: 3, 4: 3},
    },
]

def _topology_matrix(edges):
    matrix = np.eye(5, dtype=int)
    for left, right in edges:
        matrix[left, right] = 1
        matrix[right, left] = 1
    return matrix.tolist()

def _compatible_os(service, rng):
    if service == "ssh":
        return "linux"
    if service == "ftp":
        return "windows"
    return str(rng.choice(OS_VOCAB))

def build_random_scenario(seed, difficulty, split):
    rng = np.random.default_rng(seed)
    user_hosts = int(rng.integers(2, 4 + difficulty))
    user_hosts = min(user_hosts, 5)
    subnets = [1, 1, user_hosts, 1]

    if difficulty == 0:
        template_index = 0
    else:
        template_index = int(rng.integers(0, len(TOPOLOGY_TEMPLATES)))
    template = TOPOLOGY_TEMPLATES[template_index]
    topology = _topology_matrix(template["edges"])

    gateway_service = {
        subnet: str(rng.choice(SERVICE_VOCAB))
        for subnet in range(1, 5)
    }

    if difficulty == 0:
        exploit_probs = {service: 0.9 for service in SERVICE_VOCAB}
        privesc_prob = 1.0
        exploit_costs = {service: 1.0 for service in SERVICE_VOCAB}
    elif difficulty == 1:
        exploit_probs = {
            service: float(rng.choice([0.65, 0.8, 0.9]))
            for service in SERVICE_VOCAB
        }
        privesc_prob = float(rng.choice([0.8, 0.9, 1.0]))
        exploit_costs = {
            service: float(rng.choice([1.0, 2.0]))
            for service in SERVICE_VOCAB
        }
    else:
        exploit_probs = {
            service: float(rng.choice([0.4, 0.55, 0.7, 0.9]))
            for service in SERVICE_VOCAB
        }
        privesc_prob = float(rng.choice([0.7, 0.85, 1.0]))
        exploit_costs = {
            service: float(rng.choice([1.0, 2.0, 3.0]))
            for service in SERVICE_VOCAB
        }

    firewall = {}
    for left, right in template["edges"]:
        for src, dst in ((left, right), (right, left)):
            if difficulty == 0:
                allowed = list(SERVICE_VOCAB)
            else:
                count = 2 if difficulty == 1 else int(rng.integers(1, 3))
                allowed = [
                    str(service)
                    for service in rng.choice(
                        SERVICE_VOCAB, size=count, replace=False
                    )
                ]
            if template["parents"].get(dst) == src:
                allowed = list(dict.fromkeys(allowed + [gateway_service[dst]]))
            firewall[str((src, dst))] = allowed

    addresses = [(1, 0), (2, 0)]
    addresses.extend((3, host_id) for host_id in range(user_hosts))
    addresses.append((4, 0))
    sensitive_hosts = {(2, 0): 20, (4, 0): 20}
    honeypot_candidates = [(3, host_id) for host_id in range(user_hosts)]
    honeypot = honeypot_candidates[int(rng.integers(0, len(honeypot_candidates)))]

    host_configurations = {}
    for address in addresses:
        subnet = address[0]
        required_service = gateway_service[subnet]
        host_os = _compatible_os(required_service, rng)

        services = [required_service]
        extra_count = int(rng.integers(0, 1 + difficulty))
        extra_count = min(extra_count, len(SERVICE_VOCAB) - 1)
        extras = [
            str(service)
            for service in rng.choice(
                SERVICE_VOCAB, size=extra_count, replace=False
            )
        ]
        services = list(dict.fromkeys(services + extras))

        required_process = "tomcat" if host_os == "linux" else "daclsvc"
        processes = [required_process]
        if difficulty >= 1 and rng.random() < 0.25:
            processes = list(PROCESS_VOCAB)

        if address in sensitive_hosts:
            value = sensitive_hosts[address]
        elif address == honeypot:
            value = -8
        else:
            value = int(rng.integers(2, 8))

        host_configurations[str(address)] = {
            "os": host_os,
            "services": services,
            "processes": processes,
            "value": value,
        }

    scenario = {
        "subnets": subnets,
        "topology": topology,
        "sensitive_hosts": {
            str(address): value for address, value in sensitive_hosts.items()
        },
        "os": list(OS_VOCAB),
        "services": list(SERVICE_VOCAB),
        "processes": list(PROCESS_VOCAB),
        "exploits": {
            "e_ssh": {
                "service": "ssh", "os": "linux",
                "prob": exploit_probs["ssh"],
                "cost": exploit_costs["ssh"], "access": "user",
            },
            "e_ftp": {
                "service": "ftp", "os": "windows",
                "prob": exploit_probs["ftp"],
                "cost": exploit_costs["ftp"], "access": "user",
            },
            "e_http": {
                "service": "http", "os": "None",
                "prob": exploit_probs["http"],
                "cost": exploit_costs["http"], "access": "user",
            },
        },
        "privilege_escalation": {
            "pe_tomcat": {
                "process": "tomcat", "os": "linux",
                "prob": privesc_prob, "cost": 1.0, "access": "root",
            },
            "pe_daclsvc": {
                "process": "daclsvc", "os": "windows",
                "prob": privesc_prob, "cost": 1.0, "access": "root",
            },
        },
        "service_scan_cost": 0.0,
        "os_scan_cost": 0.0,
        "subnet_scan_cost": 0.0,
        "process_scan_cost": 0.0,
        "host_configurations": host_configurations,
        "firewall": firewall,
        "step_limit": 400,
    }
    return scenario, honeypot

def create_scenario_bank(split, count_by_difficulty, seed_offset):
    specs = []
    scenario_number = 0
    for difficulty, count in enumerate(count_by_difficulty):
        for local_index in range(count):
            seed = seed_offset + difficulty * 1000 + local_index
            scenario, honeypot = build_random_scenario(seed, difficulty, split)
            path = SCENARIO_DIR / f"{split}_d{difficulty}_{local_index:03d}.yaml"
            with path.open("w") as handle:
                yaml.safe_dump(scenario, handle, sort_keys=False)
            specs.append({
                "path": str(path),
                "split": split,
                "difficulty": difficulty,
                "seed": seed,
                "honeypot": honeypot,
                "name": f"{split}-d{difficulty}-{local_index:03d}",
            })
            scenario_number += 1
    return specs

# Increase these counts for a larger domain-randomization bank.
TRAIN_SPECS = create_scenario_bank("train", [12, 12, 12], 10_000)
VALIDATION_SPECS = create_scenario_bank("validation", [3, 4, 5], 50_000)
TEST_SPECS = create_scenario_bank("test", [5, 5, 10], 90_000)

print(
    f"Generated {len(TRAIN_SPECS)} training, "
    f"{len(VALIDATION_SPECS)} validation, and {len(TEST_SPECS)} test scenarios."
)


## 4. Action outcome classification

In [ ]:
class ActionOutcome(Enum):
    SUCCESS = "success"
    MISMATCH = "mismatch"
    BLOCKED = "blocked"
    PROB_FAIL = "prob_fail"
    PERMISSION = "permission"
    HONEYPOT = "honeypot"

def classify_outcome(action, info, env):
    if info.get("success", False):
        return ActionOutcome.SUCCESS
    target = action.target
    service = action.service
    if not env.current_state.host_is_running_service(target, service):
        return ActionOutcome.MISMATCH
    if not env.network.traffic_permitted(env.current_state, target, service):
        return ActionOutcome.BLOCKED
    if info.get("permission_error", False):
        return ActionOutcome.MISMATCH   
    return ActionOutcome.PROB_FAIL

def classify_outcome_privesc(action, info, env):
    if info.get("success", False):
        return ActionOutcome.SUCCESS
    target = action.target
    process = action.process
    host = env.current_state.get_host(target)
    if not host.is_running_process(process):
        return ActionOutcome.MISMATCH
    if action.os is not None and not host.is_running_os(action.os):
        return ActionOutcome.MISMATCH
    if host.access < action.req_access:
        return ActionOutcome.PERMISSION
    return ActionOutcome.PROB_FAIL

## 5. Load a reference environment and validate the scenario schema

All splits must expose the same OS/service/process/action vocabulary. Host count and topology may differ.


In [ ]:
def load_scenario_spec(spec):
    return nasim.load(spec["path"], fully_obs=False)

def scenario_signature(env):
    return (
        tuple(env.scenario.os),
        tuple(env.scenario.services),
        tuple(env.scenario.processes),
        tuple(env.scenario.exploits.keys()),
        tuple(env.scenario.privescs.keys()),
    )

reference_spec = TRAIN_SPECS[0]
env = load_scenario_spec(reference_spec)
reference_signature = scenario_signature(env)

for spec in TRAIN_SPECS + VALIDATION_SPECS + TEST_SPECS:
    candidate_env = load_scenario_spec(spec)
    if scenario_signature(candidate_env) != reference_signature:
        raise ValueError(f"Incompatible scenario vocabulary: {spec['name']}")

# Reload after validation because HostVector uses scenario-dependent class metadata.
env = load_scenario_spec(reference_spec)
obs, info = env.reset()

service_list = list(env.scenario.services)
service_to_idx = {name: i for i, name in enumerate(service_list)}
PROCESS_LIST = list(env.scenario.processes)
PROCESS_TO_IDX = {name: i for i, name in enumerate(PROCESS_LIST)}

print("Scenario schema validated:", reference_signature)


## 6. Environment step (native NASim reward)

The baseline uses NASim's reward unchanged. IDS and explicit honeypot shaping remain optional and are disabled below.


In [ ]:
def step_env(env, action):
    # Keep the baseline reward interpretable: NASim value/discovery rewards minus action cost.
    return env.step(action)


In [ ]:
def get_reference_value(env, spec):
    """A per-scenario baseline so bonus floors scale with this network, not a hardcoded number."""
    values = [
        env.current_state.get_host(addr).value
        for addr in env.current_state.host_num_map
        if addr != spec["honeypot"]
    ]
    return max(sum(values) / len(values), 1.0) if values else 1.0

SCAN_INFORMED_EXPLOIT_BONUS = 1.25
BLIND_EXPLOIT_PENALTY = -1.75
SCAN_INFORMED_PRIVESC_BONUS = 2.5
BLIND_PRIVESC_PENALTY = -3.5

def compute_reward(env, action, info, outcome, current_spec,
                    was_compromised, had_root_access, reference_value,
                    reward_from_env=0.0,
                    known_before_service=None, known_before_process=None):
    reward = reward_from_env - info.get("value", 0.0)  # strip native value credit — avoid double count
    components = {"base_cost": reward}

    outcome_penalties = {
        ActionOutcome.SUCCESS: 0.0,
        ActionOutcome.MISMATCH: -0.2,
        ActionOutcome.BLOCKED: -0.1,
        ActionOutcome.PROB_FAIL: -0.05,
        ActionOutcome.PERMISSION: -0.15,
    }
    outcome_pen = outcome_penalties.get(outcome, 0.0)
    if outcome_pen != 0.0:
        reward += outcome_pen
        components["outcome_penalty"] = outcome_pen

    host_value = env.current_state.get_host(action.target).value if hasattr(action, "target") else 0
    is_priv = isinstance(action, PrivilegeEscalation)

    if action.is_exploit() and info.get("success") and not was_compromised:
        bonus = max(host_value * 1.5, 0.5 * reference_value)
        reward += bonus
        components["exploit_success_bonus"] = bonus
    if is_priv and info.get("success") and not had_root_access:
        bonus = max(host_value * 1.5, 0.7 * reference_value)
        reward += bonus
        components["privesc_success_bonus"] = bonus

    if action.is_exploit() and known_before_service is not None:
        if known_before_service == 1:
            reward += SCAN_INFORMED_EXPLOIT_BONUS
            components["scan_informed_exploit_bonus"] = SCAN_INFORMED_EXPLOIT_BONUS
        elif known_before_service == 0:
            reward += BLIND_EXPLOIT_PENALTY
            components["blind_exploit_penalty"] = BLIND_EXPLOIT_PENALTY

    if is_priv and known_before_process is not None:
        if known_before_process == 1:
            reward += SCAN_INFORMED_PRIVESC_BONUS
            components["scan_informed_privesc_bonus"] = SCAN_INFORMED_PRIVESC_BONUS
        elif known_before_process == 0:
            reward += BLIND_PRIVESC_PENALTY
            components["blind_privesc_penalty"] = BLIND_PRIVESC_PENALTY

    newly_discovered = info.get("newly_discovered", {})
    actually_new = sum(1 for v in newly_discovered.values() if v)
    if getattr(action, "name", "") == "subnet_scan" and info.get("success") and actually_new:
        bonus = 1.0 * actually_new
        reward += bonus
        components["discovery_bonus"] = bonus

    components["total"] = reward
    return reward, components


## 7. Per-episode semantic memory and action-result trackers

Only one environment is active at a time. Every episode receives fresh tracker dictionaries so knowledge never leaks between simulations.


In [ ]:
known_host_states = {}
node_service_state = {}
node_process_state = {}
node_scan_state = {}
edge_tracker = {}
host_suspicion = {}

SCAN_ACTION_NAMES = ("service_scan", "os_scan", "subnet_scan", "process_scan")

def reset_episode_memory():
    known_host_states.clear()
    node_service_state.clear()
    node_process_state.clear()
    node_scan_state.clear()
    edge_tracker.clear()
    host_suspicion.clear()

def update_node_tracker(host, service_idx, outcome):
    key = (host, service_idx)
    entry = node_service_state.get(key, {"known": 0, "fail_count": 0, "attempts": 0})
    entry["attempts"] += 1
    if outcome == ActionOutcome.SUCCESS:
        entry["known"] = 1
    elif outcome in (ActionOutcome.MISMATCH, ActionOutcome.BLOCKED):
        entry["known"] = -1
    elif outcome == ActionOutcome.PROB_FAIL:
        entry["fail_count"] += 1
    node_service_state[key] = entry
    return entry

def update_node_process_tracker(host, process_idx, outcome):
    key = (host, process_idx)
    entry = node_process_state.get(key, {"known": 0, "fail_count": 0, "attempts": 0})
    entry["attempts"] += 1
    if outcome == ActionOutcome.SUCCESS:
        entry["known"] = 1
    elif outcome == ActionOutcome.MISMATCH:
        entry["known"] = -1
    elif outcome == ActionOutcome.PROB_FAIL:
        entry["fail_count"] += 1
    node_process_state[key] = entry
    return entry

def _scan_name(action):
    if isinstance(action, ServiceScan):
        return "service_scan"
    if isinstance(action, OSScan):
        return "os_scan"
    if isinstance(action, SubnetScan):
        return "subnet_scan"
    if isinstance(action, ProcessScan):
        return "process_scan"
    return None

def _scan_outcome(info):
    if info.get("success", False):
        return ActionOutcome.SUCCESS
    if info.get("permission_error", False):
        return ActionOutcome.PERMISSION
    if info.get("connection_error", False):
        return ActionOutcome.BLOCKED
    return ActionOutcome.PROB_FAIL

def _record_service_knowledge(host, env):
    for service, service_idx in service_to_idx.items():
        entry = node_service_state.get(
            (host, service_idx), {"known": 0, "fail_count": 0, "attempts": 0}
        )
        entry["known"] = 1 if env.current_state.host_is_running_service(host, service) else -1
        node_service_state[(host, service_idx)] = entry

def _record_process_knowledge(host, env):
    host_state = env.current_state.get_host(host)
    for process, process_idx in PROCESS_TO_IDX.items():
        entry = node_process_state.get(
            (host, process_idx), {"known": 0, "fail_count": 0, "attempts": 0}
        )
        entry["known"] = 1 if host_state.is_running_process(process) else -1
        node_process_state[(host, process_idx)] = entry

def get_compromised_subnets(env):
    return {
        address[0]
        for address in env.current_state.host_num_map
        if env.current_state.host_compromised(address)
    }

def update_edge_tracker_for_action(edge_tracker, action, env, outcome):
    if outcome not in (ActionOutcome.BLOCKED, ActionOutcome.SUCCESS, ActionOutcome.PROB_FAIL):
        return
    dst_subnet = action.target[0]
    service_idx = service_to_idx[action.service]
    for src_subnet in get_compromised_subnets(env):
        if src_subnet == dst_subnet:
            continue
        permitted = env.network.subnet_traffic_permitted(
            src_subnet, dst_subnet, action.service
        )
        key = (src_subnet, dst_subnet, service_idx)
        entry = edge_tracker.get(key, {"outcome": 0, "attempts": 0})
        entry["attempts"] += 1
        entry["outcome"] = 1 if permitted else -1
        edge_tracker[key] = entry

def update_trackers(action, info, env, edge_tracker, node_service_state,
                    node_process_state, node_scan_state):
    scan_name = _scan_name(action)
    if scan_name is not None:
        outcome = _scan_outcome(info)
        if outcome == ActionOutcome.SUCCESS:
            node_scan_state[(action.target, scan_name)] = True
            if isinstance(action, ServiceScan):
                _record_service_knowledge(action.target, env)
            elif isinstance(action, ProcessScan):
                _record_process_knowledge(action.target, env)
        return outcome

    if isinstance(action, Exploit):
        outcome = classify_outcome(action, info, env)
        update_node_tracker(action.target, service_to_idx[action.service], outcome)
        update_edge_tracker_for_action(edge_tracker, action, env, outcome)
        if outcome == ActionOutcome.SUCCESS:
            _record_service_knowledge(action.target, env)
            node_scan_state[(action.target, "service_scan")] = True
            node_scan_state[(action.target, "os_scan")] = True
        return outcome

    if isinstance(action, PrivilegeEscalation):
        outcome = classify_outcome_privesc(action, info, env)
        update_node_process_tracker(
            action.target, PROCESS_TO_IDX[action.process], outcome
        )
        if outcome == ActionOutcome.SUCCESS:
            _record_process_knowledge(action.target, env)
            node_scan_state[(action.target, "process_scan")] = True
            node_scan_state[(action.target, "os_scan")] = True
        return outcome

    return None


## 8. Fixed-width semantic graph construction

Raw NASim HostVector indices move when address-space bounds change. The graph therefore uses a stable semantic representation, allowing scenarios with different host counts to share one model.


In [ ]:
def read_host_observation(host_slice, address):
    readable = HostVector.get_readable(host_slice)
    return {
        "address": address,
        "compromised": float(readable["Compromised"]),
        "reachable": float(readable["Reachable"]),
        "discovered": float(readable["Discovered"]),
        "value": float(readable["Value"]),
        "discovery_value": float(readable["Discovery Value"]),
        "access": float(readable["Access"]),
        "os": np.asarray([readable[name] for name in OS_VOCAB], dtype=np.float32),
        "services": np.asarray([readable[name] for name in SERVICE_VOCAB], dtype=np.float32),
        "processes": np.asarray([readable[name] for name in PROCESS_VOCAB], dtype=np.float32),
    }

def merge_host_observation(previous, current):
    merged = {
        key: (value.copy() if isinstance(value, np.ndarray) else value)
        for key, value in previous.items()
    }
    for key in ("compromised", "reachable", "discovered", "access"):
        merged[key] = max(previous[key], current[key])
    for key in ("value", "discovery_value"):
        if current[key] != 0:
            merged[key] = current[key]
    for key in ("os", "services", "processes"):
        if np.any(current[key] != 0):
            merged[key] = current[key].copy()
    return merged

def _tracker_features(host, state_map, count):
    features = []
    for item_idx in range(count):
        entry = state_map.get((host, item_idx), {})
        features.extend([
            float(entry.get("known", 0)),
            float(np.log1p(entry.get("fail_count", 0))),
            float(np.log1p(entry.get("attempts", 0))),
        ])
    return features

SEMANTIC_HOST_FEATURE_DIM = (
    2 + 6 + len(OS_VOCAB) + len(SERVICE_VOCAB) + len(PROCESS_VOCAB)
)
NODE_FEATURE_DIM = (
    SEMANTIC_HOST_FEATURE_DIM
    + 3 * len(SERVICE_VOCAB)
    + 3 * len(PROCESS_VOCAB)
    + len(SCAN_ACTION_NAMES)
)
EDGE_FEATURE_DIM = len(SERVICE_VOCAB) + 1

def host_feature_vector(host, memory):
    subnet, host_id = host
    base = np.concatenate([
        np.asarray([
            subnet / MAX_SUBNET_ID,
            host_id / max(MAX_HOST_ID, 1),
            memory["compromised"],
            memory["reachable"],
            memory["discovered"],
            memory["value"] / 20.0,
            memory["discovery_value"] / 20.0,
            memory["access"] / float(AccessLevel.ROOT),
        ], dtype=np.float32),
        memory["os"],
        memory["services"],
        memory["processes"],
    ])
    service_features = _tracker_features(
        host, node_service_state, len(SERVICE_VOCAB)
    )
    process_features = _tracker_features(
        host, node_process_state, len(PROCESS_VOCAB)
    )
    scan_features = [
        float(node_scan_state.get((host, scan_name), False))
        for scan_name in SCAN_ACTION_NAMES
    ]
    return np.concatenate([
        base,
        np.asarray(service_features, dtype=np.float32),
        np.asarray(process_features, dtype=np.float32),
        np.asarray(scan_features, dtype=np.float32),
    ])

def build_graph(obs, env):
    host_vec_size = HostVector.state_size
    obs_2d = obs.reshape(-1, host_vec_size)

    for idx, address in enumerate(env.scenario.address_space):
        host_slice = obs_2d[idx].copy()
        current = read_host_observation(host_slice, address)
        if current["discovered"]:
            if address not in known_host_states:
                known_host_states[address] = current
            else:
                known_host_states[address] = merge_host_observation(
                    known_host_states[address], current
                )

    discovered_hosts = list(known_host_states.keys())
    node_features = [
        host_feature_vector(address, known_host_states[address])
        for address in discovered_hosts
    ]
    host_to_idx = {address: i for i, address in enumerate(discovered_hosts)}

    edge_list = []
    edge_features = []
    for address_a in discovered_hosts:
        for address_b in discovered_hosts:
            if address_a == address_b:
                continue
            subnet_a, subnet_b = address_a[0], address_b[0]
            if env.scenario.topology[subnet_a][subnet_b] != 1:
                continue
            edge_list.append((host_to_idx[address_a], host_to_idx[address_b]))
            reachability = [
                edge_tracker.get((subnet_a, subnet_b, service_idx), {}).get("outcome", 0)
                for service_idx in range(len(SERVICE_VOCAB))
            ]
            edge_features.append(
                reachability + [1 if subnet_a == subnet_b else 0]
            )

    return {
        "discovered_hosts": discovered_hosts,
        "host_to_idx": host_to_idx,
        "node_features": np.asarray(node_features, dtype=np.float32),
        "edges": edge_list,
        "edge_features": np.asarray(edge_features, dtype=np.float32),
    }

def convert_to_PyG(graph):
    if len(graph["node_features"]) > 0:
        x = torch.tensor(graph["node_features"], dtype=torch.float32)
    else:
        x = torch.empty((0, NODE_FEATURE_DIM), dtype=torch.float32)

    if len(graph["edges"]) > 0:
        edge_index = torch.tensor(
            graph["edges"], dtype=torch.long
        ).t().contiguous()
        edge_attr = torch.tensor(
            graph["edge_features"], dtype=torch.float32
        )
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, EDGE_FEATURE_DIM), dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


## 9. GATv2 encoder + actor/critic

In [ ]:
class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim, num_heads=4, dropout=0.0):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=num_heads, concat=True,
                                edge_dim=edge_dim, dropout=dropout, bias=True)
        self.conv2 = GATv2Conv(hidden_channels * num_heads, out_channels, heads=1, concat=False,
                                edge_dim=edge_dim, dropout=dropout, bias=True)

    def forward(self, x, edge_index, edge_attr=None):
        x = self.conv1(x, edge_index, edge_attr)
        x = F.leaky_relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        return x

class NetworkStateEncoder(nn.Module):
    def __init__(self, **gnn_kwargs):
        super().__init__()
        self.gnn = GATEncoder(**gnn_kwargs)

    def forward(self, obs, env):
        graph = build_graph(obs, env)
        pyg_data = convert_to_PyG(graph)
        node_emb = self.gnn(pyg_data.x, pyg_data.edge_index, pyg_data.edge_attr)
        # Mean and max pooling provide both average state and salient-host signals.
        graph_emb = torch.cat([
            node_emb.mean(dim=0),
            node_emb.max(dim=0).values,
        ])
        return graph_emb, node_emb, graph, pyg_data

class ActorNetwork(nn.Module):
    def __init__(self, action_types_per_host, embedding_size=16):
        super().__init__()
        self.actor_layer = nn.Linear(embedding_size, action_types_per_host)

    def forward(self, node_embeddings, mask=None, deterministic=False):
        action_scores = self.actor_layer(node_embeddings).reshape(-1)
        if mask is not None:
            action_scores = action_scores.masked_fill(~mask, -1e9)
        action_distribution = Categorical(logits=action_scores)
        chosen_action = (
            torch.argmax(action_scores)
            if deterministic else action_distribution.sample()
        )
        log_prob = action_distribution.log_prob(chosen_action)
        return chosen_action, log_prob, action_distribution

class Critic(nn.Module):
    def __init__(self, embedding_size):
        super().__init__()
        self.critic_layer = nn.Linear(embedding_size, 1)

    def forward(self, graph_emb):
        return self.critic_layer(graph_emb)


In [ ]:
def compute_explained_variance(y_true, y_pred):
    """
    y_true: Empirical returns collected (returns)
    y_pred: Predicted state values from critic (values)
    """
    var_y = np.var(y_true)
    if var_y == 0:
        return 0.0  # Avoid division by zero if all returns in batch are identical
    return 1.0 - (np.var(y_true - y_pred) / var_y)

## 10. Action space, semantic mask, and decode helpers

Completed scans, exploits on already-compromised hosts, and privilege escalation without user access are masked. Action definitions come directly from the loaded YAML scenario.


In [ ]:
SCAN_ACTION_NAMES = ["service_scan", "os_scan", "subnet_scan", "process_scan"]
action_types = (
    SCAN_ACTION_NAMES
    + list(env.scenario.exploits.keys())
    + list(env.scenario.privescs.keys())
)
action_types_per_host = len(action_types)

action_type_to_service = {
    name: definition["service"]
    for name, definition in env.scenario.exploits.items()
}
action_type_to_process = {
    name: definition["process"]
    for name, definition in env.scenario.privescs.items()
}

def build_action_mask(discovered_hosts, action_types, node_service_state,
                      node_process_state, node_scan_state, known_host_states):
    mask = []
    for host in discovered_hosts:
        remembered = known_host_states[host]
        compromised = bool(remembered["compromised"])
        root_access = remembered["access"] >= AccessLevel.ROOT

        for action_name in action_types:
            if action_name in SCAN_ACTION_NAMES:
                allowed = not node_scan_state.get((host, action_name), False)
                if action_name in ("subnet_scan", "process_scan"):
                    allowed = allowed and compromised
                mask.append(allowed)
            elif action_name in action_type_to_service:
                service_idx = service_to_idx[action_type_to_service[action_name]]
                known = node_service_state.get((host, service_idx), {}).get("known", 0)
                mask.append((not compromised) and known != -1)
            elif action_name in action_type_to_process:
                process_idx = PROCESS_TO_IDX[action_type_to_process[action_name]]
                known = node_process_state.get((host, process_idx), {}).get("known", 0)
                mask.append(compromised and (not root_access) and known != -1)
            else:
                mask.append(False)

    mask = torch.tensor(mask, dtype=torch.bool)
    if not mask.any():
        raise RuntimeError("Action mask contains no valid actions; inspect tracker state.")
    return mask

def build_nasim_action(action_type_name, target, env):
    scan_costs = {
        "service_scan": env.scenario.service_scan_cost,
        "os_scan": env.scenario.os_scan_cost,
        "subnet_scan": env.scenario.subnet_scan_cost,
        "process_scan": env.scenario.process_scan_cost,
    }
    scan_classes = {
        "service_scan": ServiceScan,
        "os_scan": OSScan,
        "subnet_scan": SubnetScan,
        "process_scan": ProcessScan,
    }
    if action_type_name in scan_classes:
        return scan_classes[action_type_name](
            target=target, cost=scan_costs[action_type_name]
        )
    if action_type_name in env.scenario.exploits:
        return Exploit(
            name=action_type_name,
            target=target,
            **env.scenario.exploits[action_type_name],
        )
    if action_type_name in env.scenario.privescs:
        return PrivilegeEscalation(
            name=action_type_name,
            target=target,
            **env.scenario.privescs[action_type_name],
        )
    raise ValueError(f"Unknown action type: {action_type_name}")


## 11. IDS suspicion tracking (currently unused — `ids_enabled = False` below)

In [ ]:
NOISE_WEIGHT = {"scan": 0.01, "exploit": 0.05, "privesc": 0.07}
OUTCOME_NOISE_MULTIPLIER = {
    ActionOutcome.SUCCESS: 1.0,
    ActionOutcome.MISMATCH: 0.5,
    ActionOutcome.BLOCKED: 0.3,
    ActionOutcome.PROB_FAIL: 0.7,
    ActionOutcome.PERMISSION: 1.0,
}
DETECTION_THRESHOLD = 1.5

def action_kind(action):
    if hasattr(action, "service") and action.service is not None:
        return "exploit"
    if hasattr(action, "process") and action.process is not None:
        return "privesc"
    return "scan"

def update_ids(host_suspicion, action, outcome, node_service_state, node_process_state):
    host = action.target
    kind = action_kind(action)
    base = NOISE_WEIGHT[kind]
    multiplier = OUTCOME_NOISE_MULTIPLIER.get(outcome, 1.0)

    repeat_count = 1
    if kind == "exploit":
        key = (host, service_to_idx[action.service])
        repeat_count = node_service_state.get(key, {}).get("attempts", 1)
    elif kind == "privesc":
        key = (host, PROCESS_TO_IDX[action.process])
        repeat_count = node_process_state.get(key, {}).get("attempts", 1)

    escalation = 1 + (repeat_count - 1) * 0.1
    increment = base * multiplier * escalation
    host_suspicion[host] = host_suspicion.get(host, 0.0) + increment
    return host_suspicion[host]

def check_ids_termination(host_suspicion):
    for host, score in host_suspicion.items():
        if score >= DETECTION_THRESHOLD:
            return True, host
    return False, None

def compute_ids_penalty(host_suspicion, host, outcome):
    if outcome in (ActionOutcome.MISMATCH, ActionOutcome.PROB_FAIL, ActionOutcome.PERMISSION):
        return -0.5 * host_suspicion.get(host, 0.0)
    return 0.0

## 12. Honeypot check

The honeypot location belongs to the current scenario specification and changes between episodes.


In [ ]:
honeypot_penalty = -5.0

def check_honeypot(action, honeypot_host):
    if action.target != honeypot_host:
        return False
    return action.is_exploit() or isinstance(action, PrivilegeEscalation)


## 13. Rollout buffer, bootstrapped GAE, and PPO minibatch loss


In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.observations = []
        self.actions = []
        self.log_probs = []
        self.values = []
        self.rewards = []
        self.dones = []
        self.pyg_data_list = []
        self.masks = []

    def store(self, obs, action, log_prob, value, reward, done, pyg_data, mask):
        self.observations.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.rewards.append(reward)
        self.dones.append(done)
        self.pyg_data_list.append(pyg_data)
        self.masks.append(mask)

    def clear(self):
        self.__init__()

    def __len__(self):
        return len(self.rewards)

def compute_returns_and_advantages(buffer, last_value, gamma=0.99, lam=0.95):
    values = torch.stack(buffer.values).reshape(-1)
    rewards = torch.as_tensor(buffer.rewards, dtype=values.dtype, device=values.device)
    dones = torch.as_tensor(buffer.dones, dtype=values.dtype, device=values.device)
    last_value = torch.as_tensor(last_value, dtype=values.dtype, device=values.device).detach()

    advantages = torch.zeros_like(values)
    last_gae = torch.zeros((), dtype=values.dtype, device=values.device)

    for t in reversed(range(len(buffer))):
        next_value = last_value if t == len(buffer) - 1 else values[t + 1]
        next_non_terminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
        last_gae = delta + gamma * lam * next_non_terminal * last_gae
        advantages[t] = last_gae

    returns = advantages + values
    advantages = (advantages - advantages.mean()) / (
        advantages.std(unbiased=False) + 1e-8
    )
    return returns.detach(), advantages.detach()

def compute_ppo_loss(actor, critic, state_encoder, buffer, returns, advantages,
                     batch_indices, epsilon=0.2, ent_coef=0.01):
    indices = [int(idx) for idx in batch_indices]
    old_log_probs = torch.stack(
        [buffer.log_probs[idx] for idx in indices]
    ).reshape(-1).detach()

    new_log_probs, new_values, entropies = [], [], []
    for idx in indices:
        pyg_data = buffer.pyg_data_list[idx]
        action = buffer.actions[idx]
        mask = buffer.masks[idx]

        node_emb = state_encoder.gnn(pyg_data.x, pyg_data.edge_index, pyg_data.edge_attr)
        graph_emb = torch.cat([
            node_emb.mean(dim=0),
            node_emb.max(dim=0).values,
        ])
        _, _, dist = actor(node_emb, mask)
        new_log_probs.append(dist.log_prob(action))
        entropies.append(dist.entropy())
        new_values.append(critic(graph_emb))

    new_log_probs = torch.stack(new_log_probs).reshape(-1)
    entropies = torch.stack(entropies).reshape(-1)
    new_values = torch.stack(new_values).reshape(-1)

    batch_returns = returns[indices]
    batch_advantages = advantages[indices]
    ratio = torch.exp(new_log_probs - old_log_probs)
    unclipped = ratio * batch_advantages
    clipped = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * batch_advantages
    actor_loss = -torch.min(unclipped, clipped).mean()

    critic_loss = F.mse_loss(new_values, batch_returns)
    entropy_bonus = entropies.mean()
    return actor_loss + 0.50 * critic_loss - ent_coef * entropy_bonus


## 14. Instantiate models

In [ ]:
state_encoder = NetworkStateEncoder(
    in_channels=NODE_FEATURE_DIM,
    hidden_channels=16,
    out_channels=16,
    edge_dim=EDGE_FEATURE_DIM,
    num_heads=4,
)
actor = ActorNetwork(action_types_per_host, embedding_size=16)
critic = Critic(32)  # mean + max pooled 16-dimensional graph embeddings

trainable_parameters = (
    list(state_encoder.parameters())
    + list(actor.parameters())
    + list(critic.parameters())
)
optimizer = torch.optim.Adam(trainable_parameters, lr=3e-4)


In [ ]:
import os
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(iteration, tag=None):
    name = f"checkpoint_iter{iteration}" if tag is None else f"checkpoint_{tag}"
    path = os.path.join(CHECKPOINT_DIR, f"{name}.pt")
    torch.save({
        "iteration": iteration,
        "state_encoder": state_encoder.state_dict(),
        "actor": actor.state_dict(),
        "critic": critic.state_dict(),
        "optimizer": optimizer.state_dict(),
        "reward_history": reward_history,
        "loss_history": loss_history,
        "episode_length_history": episode_length_history,
        "goal_success_rate_history": goal_success_rate_history,
        "coverage_history": coverage_history,
        "honeypot_rate_history": honeypot_rate_history,
        "validation_history": validation_history,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "sampling_rng_state": sampling_rng.bit_generator.state,
    }, path)
    print(f"Saved checkpoint: {path}")

In [ ]:
def load_checkpoint(path):
    ckpt = torch.load(path, map_location="cpu")
    state_encoder.load_state_dict(ckpt["state_encoder"])
    actor.load_state_dict(ckpt["actor"])
    critic.load_state_dict(ckpt["critic"])
    optimizer.load_state_dict(ckpt["optimizer"])
    reward_history[:] = ckpt["reward_history"]
    loss_history[:] = ckpt["loss_history"]
    episode_length_history[:] = ckpt["episode_length_history"]
    goal_success_rate_history[:] = ckpt["goal_success_rate_history"]
    coverage_history[:] = ckpt["coverage_history"]
    honeypot_rate_history[:] = ckpt["honeypot_rate_history"]
    validation_history[:] = ckpt["validation_history"]
    torch.set_rng_state(ckpt["torch_rng_state"])
    np.random.set_state(ckpt["numpy_rng_state"])
    sampling_rng.bit_generator.state = ckpt["sampling_rng_state"]
    print(f"Loaded checkpoint from iteration {ckpt['iteration']}")
    return ckpt["iteration"]

## 15. Curriculum-based multi-scenario training

Each completed episode samples another training network. Early iterations use easier scenarios; medium and hard scenarios are introduced progressively. Validation runs use held-out networks and never update the model.


In [ ]:
def evaluate_scenario_specs(specs, deterministic=True):
    results = []
    for spec in specs:
        eval_env, eval_obs, _, required_hosts = start_episode(spec)
        reference_value = get_reference_value(eval_env, spec)  # computed ONCE per episode, uses `spec` not `current_spec`
        total_reward = 0.0
        touched_honeypot = False
        terminated = False
        steps_taken = 0

        for steps_taken in range(1, eval_env.scenario.step_limit + 1):
            with torch.no_grad():
                action, _, _, _, _, _ = select_policy_action(
                    eval_obs, eval_env, deterministic=deterministic
                )

            was_compromised = False
            had_root_access = False
            known_before_service = None
            known_before_process = None
            if action.is_exploit():
                host_before = eval_env.current_state.get_host(action.target)
                was_compromised = bool(host_before.compromised)
                svc_idx = service_to_idx[action.service]
                known_before_service = node_service_state.get((action.target, svc_idx), {}).get("known", 0)
            elif isinstance(action, PrivilegeEscalation):
                host_before = eval_env.current_state.get_host(action.target)
                had_root_access = (host_before.access >= AccessLevel.ROOT)
                proc_idx = PROCESS_TO_IDX[action.process]
                known_before_process = node_process_state.get((action.target, proc_idx), {}).get("known", 0)

            next_obs, native_reward, terminated, truncated, info = step_env(eval_env, action)
            outcome = update_trackers(
                action, info, eval_env, edge_tracker,
                node_service_state, node_process_state, node_scan_state,
            )
            reward, _ = compute_reward(
                eval_env, action, info, outcome, spec,  # `spec` not `current_spec`
                was_compromised, had_root_access, reference_value,
                reward_from_env=native_reward,
                known_before_service=known_before_service,
                known_before_process=known_before_process,
            )
            # (no duplicate/second compute_reward call here — that line is deleted)

            touched_honeypot |= check_honeypot(action, spec["honeypot"])
            total_reward += reward
            eval_obs = next_obs
            if terminated or truncated:
                break

        compromised = sum(
            eval_env.current_state.host_compromised(address)
            for address in required_hosts
        )
        results.append({
            "goal_success": float(terminated),
            "coverage": compromised / len(required_hosts),
            "honeypot_hit": float(touched_honeypot),
            "reward": float(total_reward),
            "length": steps_taken,
            "difficulty": spec["difficulty"],
        })

    return {
        key: float(np.mean([result[key] for result in results]))
        for key in ("goal_success", "coverage", "honeypot_hit", "reward", "length")
    }


ids_enabled = False
honeypot_enabled = False

num_iterations = 100
rollout_steps = 1024
ppo_epochs = 4
minibatch_size = 256
ent_coef = 0.03
validation_interval = 10
validation_scenarios_per_check = 6

sampling_rng = np.random.default_rng(GLOBAL_SEED + 1)
rollout_buffer = RolloutBuffer()
reward_history = []
loss_history = []
episode_length_history = []
goal_success_rate_history = []
coverage_history = []
honeypot_rate_history = []
validation_history = []

def curriculum_max_difficulty(iteration, total_iterations):
    progress = (iteration + 1) / total_iterations
    if progress < 0.20:
        return 0
    if progress < 0.50:
        return 1
    return 2

def sample_training_spec(iteration):
    max_difficulty = curriculum_max_difficulty(iteration, num_iterations)
    eligible = [
        spec for spec in TRAIN_SPECS
        if spec["difficulty"] <= max_difficulty
    ]
    return eligible[int(sampling_rng.integers(0, len(eligible)))]

def start_episode(spec):
    episode_env = load_scenario_spec(spec)
    episode_obs, episode_info = episode_env.reset()
    reset_episode_memory()
    required_hosts = [
        address for address in episode_env.current_state.host_num_map
        if address != spec["honeypot"]
    ]
    return episode_env, episode_obs, episode_info, required_hosts

def select_policy_action(obs, env, deterministic=False):
    graph_emb, node_embeddings, graph, pyg_data = state_encoder(obs, env)
    discovered_hosts = graph["discovered_hosts"]
    mask = build_action_mask(
        discovered_hosts,
        action_types,
        node_service_state,
        node_process_state,
        node_scan_state,
        known_host_states,
    )
    chosen_action, log_prob, action_dist = actor(
        node_embeddings, mask, deterministic=deterministic
    )
    host_idx = chosen_action.item() // action_types_per_host
    action_type_idx = chosen_action.item() % action_types_per_host
    target = discovered_hosts[host_idx]
    action_name = action_types[action_type_idx]
    action = build_nasim_action(action_name, target, env)
    return action, chosen_action, log_prob, graph_emb, pyg_data, mask


current_spec = sample_training_spec(0)
env, obs, info, required_hosts = start_episode(current_spec)
reference_value = get_reference_value(env, current_spec)  # NEW: computed once at episode start, not per-step
current_ep_length = 0
current_ep_reward = 0.0
current_ep_honeypot_hit = False

for iteration in range(num_iterations):
    rollout_buffer.clear()
    episode_lengths_this_iter = []
    episode_success_this_iter = []
    episode_coverage_this_iter = []
    episode_honeypot_this_iter = []

    current_spec = sample_training_spec(iteration)
    env, obs, info, required_hosts = start_episode(current_spec)
    reference_value = get_reference_value(env, current_spec)  # NEW: recomputed for the new episode
    current_ep_length = 0
    current_ep_reward = 0.0
    current_ep_honeypot_hit = False

    for step in range(rollout_steps):
        with torch.no_grad():
            next_action, chosen_action, log_prob, graph_emb, pyg_data, mask = (
                select_policy_action(obs, env, deterministic=False)
            )
            value = critic(graph_emb).squeeze()

        was_compromised = False
        had_root_access = False
        known_before_service = None
        known_before_process = None
        if next_action.is_exploit():
            host_before = env.current_state.get_host(next_action.target)
            was_compromised = bool(host_before.compromised)
            svc_idx = service_to_idx[next_action.service]
            known_before_service = node_service_state.get((next_action.target, svc_idx), {}).get("known", 0)
        elif isinstance(next_action, PrivilegeEscalation):
            host_before = env.current_state.get_host(next_action.target)
            had_root_access = (host_before.access >= AccessLevel.ROOT)
            proc_idx = PROCESS_TO_IDX[next_action.process]
            known_before_process = node_process_state.get((next_action.target, proc_idx), {}).get("known", 0)

        next_obs, native_reward, terminated, truncated, info = step_env(env, next_action)
        outcome = update_trackers(
            next_action, info, env, edge_tracker,
            node_service_state, node_process_state, node_scan_state,
        )
        reward, _ = compute_reward(
            env, next_action, info, outcome, current_spec,
            was_compromised, had_root_access, reference_value,  # no longer undefined — set at episode start
            reward_from_env=native_reward,
            known_before_service=known_before_service,
            known_before_process=known_before_process,
        )
        # (deleted: the stray `reference_value = get_reference_value(...)` that used to live here)

        if ids_enabled:
            update_ids(
                host_suspicion, next_action, outcome,
                node_service_state, node_process_state,
            )
            reward += compute_ids_penalty(
                host_suspicion, next_action.target, outcome
            )

        targeted_honeypot = check_honeypot(
            next_action, current_spec["honeypot"]
        )
        current_ep_honeypot_hit |= targeted_honeypot
        if honeypot_enabled and targeted_honeypot:
            reward += honeypot_penalty

        current_ep_reward += reward
        ids_triggered, flagged_host = (
            check_ids_termination(host_suspicion)
            if ids_enabled else (False, None)
        )
        done = (
            ids_triggered
            or (honeypot_enabled and targeted_honeypot)
            or terminated
            or truncated
        )

        rollout_buffer.store(
            obs, chosen_action.detach(), log_prob.detach(), value.detach(),
            float(reward), bool(done), pyg_data, mask,
        )
        obs = next_obs
        current_ep_length += 1

        if done:
            compromised = sum(
                env.current_state.host_compromised(address)
                for address in required_hosts
            )
            episode_lengths_this_iter.append(current_ep_length)
            episode_success_this_iter.append(float(terminated))
            episode_coverage_this_iter.append(
                compromised / len(required_hosts)
            )
            episode_honeypot_this_iter.append(
                float(current_ep_honeypot_hit)
            )

            current_spec = sample_training_spec(iteration)
            env, obs, info, required_hosts = start_episode(current_spec)
            reference_value = get_reference_value(env, current_spec)  # NEW: recomputed for the new episode
            current_ep_length = 0
            current_ep_reward = 0.0
            current_ep_honeypot_hit = False

    with torch.no_grad():
        if rollout_buffer.dones[-1]:
            last_value = torch.tensor(0.0)
        else:
            final_graph_emb, _, _, _ = state_encoder(obs, env)
            last_value = critic(final_graph_emb).squeeze()

    returns, advantages = compute_returns_and_advantages(
        rollout_buffer, last_value, gamma=0.99, lam=0.95
    )
    y_true = returns.cpu().numpy()
    y_pred = torch.tensor(rollout_buffer.values).cpu().numpy()

    explained_var = compute_explained_variance(y_true, y_pred)

    minibatch_losses = []
    for _ in range(ppo_epochs):
        permutation = torch.randperm(len(rollout_buffer))
        for start in range(0, len(rollout_buffer), minibatch_size):
            batch_indices = permutation[start:start + minibatch_size]
            loss = compute_ppo_loss(
                actor, critic, state_encoder, rollout_buffer,
                returns, advantages, batch_indices,
                epsilon=0.2, ent_coef=ent_coef,
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=0.5)
            optimizer.step()
            minibatch_losses.append(loss.item())

    mean_loss = float(np.mean(minibatch_losses))
    mean_reward = float(np.mean(rollout_buffer.rewards))
    mean_length = float(np.mean(episode_lengths_this_iter)) if episode_lengths_this_iter else float("nan")
    goal_success = float(np.mean(episode_success_this_iter)) if episode_success_this_iter else 0.0
    mean_coverage = float(np.mean(episode_coverage_this_iter)) if episode_coverage_this_iter else 0.0
    honeypot_rate = float(np.mean(episode_honeypot_this_iter)) if episode_honeypot_this_iter else 0.0

    reward_history.append(mean_reward)
    loss_history.append(mean_loss)
    episode_length_history.append(mean_length)
    goal_success_rate_history.append(goal_success)
    coverage_history.append(mean_coverage)
    honeypot_rate_history.append(honeypot_rate)

    print(
        f"iteration {iteration}, curriculum<=d{curriculum_max_difficulty(iteration, num_iterations)}, "
        f"loss: {mean_loss:.4f}, reward: {mean_reward:.3f}, "
        f"episodes: {len(episode_lengths_this_iter)}, "
        f"explained_var: {explained_var:.3f}, "
        f"goal success: {goal_success:.1%}, coverage: {mean_coverage:.1%}, "
        f"honeypot touched: {honeypot_rate:.1%}"
    )

    if (iteration + 1) % validation_interval == 0:
        validation_subset = VALIDATION_SPECS[:validation_scenarios_per_check]
        validation_metrics = evaluate_scenario_specs(validation_subset)
        validation_metrics["iteration"] = iteration + 1
        validation_history.append(validation_metrics)

        print(
            "  held-out validation — "
            f"success: {validation_metrics['goal_success']:.1%}, "
            f"coverage: {validation_metrics['coverage']:.1%}, "
            f"honeypot touched: {validation_metrics['honeypot_hit']:.1%}"
        )
        save_checkpoint(iteration + 1)

In [ ]:
from collections import defaultdict

def evaluate_scenario_specs_by_difficulty(specs, deterministic=True):
    grouped = defaultdict(list)
    for spec in specs:
        grouped[spec["difficulty"]].append(spec)

    breakdown = {}
    for difficulty in sorted(grouped):
        metrics = evaluate_scenario_specs(grouped[difficulty], deterministic=deterministic)
        metrics["n_scenarios"] = len(grouped[difficulty])
        breakdown[difficulty] = metrics
    return breakdown

def print_difficulty_breakdown(breakdown, label="test"):
    print(f"{label} results by difficulty tier")
    print(f"{'difficulty':>10} | {'n':>3} | {'success':>8} | {'coverage':>8} | {'honeypot':>8} | {'reward':>8} | {'length':>7}")
    for difficulty, m in breakdown.items():
        print(
            f"{difficulty:>10} | {m['n_scenarios']:>3} | "
            f"{m['goal_success']:>7.1%} | {m['coverage']:>7.1%} | "
            f"{m['honeypot_hit']:>7.1%} | {m['reward']:>8.2f} | {m['length']:>7.1f}"
        )

In [ ]:
from collections import defaultdict

def describe_action(action):
    scan_name = _scan_name(action)
    if scan_name is not None:
        return f"{scan_name}({action.target})"
    if isinstance(action, Exploit):
        return f"exploit:{action.name}(target={action.target}, service={action.service})"
    if isinstance(action, PrivilegeEscalation):
        return f"privesc:{action.name}(target={action.target}, process={action.process})"
    return str(action)

def run_traced_episode(spec, deterministic=True, verbose=True):
    eval_env, eval_obs, _, required_hosts = start_episode(spec)
    total_reward = 0.0
    touched_honeypot = False
    terminated = truncated = False
    steps_taken = 0
    component_totals = defaultdict(float)
    reference_value = get_reference_value(eval_env, spec)

    for steps_taken in range(1, eval_env.scenario.step_limit + 1):
        with torch.no_grad():
            action, _, _, _, _, _ = select_policy_action(
                eval_obs, eval_env, deterministic=deterministic
            )

        was_compromised = False
        had_root_access = False
        known_before_service = None
        known_before_process = None
        if action.is_exploit():
            host_before = eval_env.current_state.get_host(action.target)
            was_compromised = bool(host_before.compromised)
            svc_idx = service_to_idx[action.service]
            known_before_service = node_service_state.get((action.target, svc_idx), {}).get("known", 0)
        elif isinstance(action, PrivilegeEscalation):
            host_before = eval_env.current_state.get_host(action.target)
            had_root_access = (host_before.access >= AccessLevel.ROOT)
            proc_idx = PROCESS_TO_IDX[action.process]
            known_before_process = node_process_state.get((action.target, proc_idx), {}).get("known", 0)

        next_obs, native_reward, terminated, truncated, info = step_env(eval_env, action)
        outcome = update_trackers(
            action, info, eval_env, edge_tracker,
            node_service_state, node_process_state, node_scan_state,
        )

        reward, components = compute_reward(
            eval_env, action, info, outcome, spec,
            was_compromised, had_root_access, reference_value,
            reward_from_env=native_reward,
            known_before_service=known_before_service,
            known_before_process=known_before_process,
        )

        if ids_enabled:
            update_ids(host_suspicion, action, outcome, node_service_state, node_process_state)
            ids_pen = compute_ids_penalty(host_suspicion, action.target, outcome)
            reward += ids_pen
            if ids_pen != 0.0:
                components["ids_penalty"] = ids_pen

        targeted_honeypot = check_honeypot(action, spec["honeypot"])
        touched_honeypot |= targeted_honeypot
        if honeypot_enabled and targeted_honeypot:
            reward += honeypot_penalty
            components["honeypot_penalty"] = honeypot_penalty
        components["total"] = reward

        for k, v in components.items():
            if k != "total":
                component_totals[k] += v
        total_reward += reward

        if verbose:
            extras = ", ".join(
                f"{k}={v:+.2f}" for k, v in components.items()
                if k not in ("base", "total")
            )
            print(
                f"step {steps_taken:3d} | {describe_action(action):45s} | "
                f"outcome={(outcome.name if outcome else '-'):10s} | "
                f"reward={reward:+6.2f} | cum={total_reward:+7.2f}"
                + (f" | {extras}" if extras else "")
            )

        eval_obs = next_obs
        if terminated or truncated:
            break

    compromised = sum(eval_env.current_state.host_compromised(a) for a in required_hosts)
    summary = {
        "goal_success": float(terminated and eval_env.goal_reached()),
        "coverage": compromised / len(required_hosts),
        "honeypot_hit": float(touched_honeypot),
        "reward": float(total_reward),
        "length": steps_taken,
        "component_totals": dict(component_totals),
    }
    if verbose:
        print("\n--- episode summary ---")
        for k, v in summary.items():
            if k != "component_totals":
                print(f"  {k}: {v}")
        print("  reward component totals:")
        for k, v in sorted(summary["component_totals"].items(), key=lambda kv: -abs(kv[1])):
            print(f"    {k}: {v:+.2f}")
        print()
    return summary

traced_specs = [
    VALIDATION_SPECS[0],
    VALIDATION_SPECS[3],
    VALIDATION_SPECS[7],
]
traced_results = []
for spec in traced_specs:
    print(f"=== spec: {spec['path']} (difficulty {spec['difficulty']}) ===")
    result = run_traced_episode(spec, deterministic=True, verbose=True)
    traced_results.append(result)
# Run on the same held-out networks each time you call this, so checkpoints are comparable
traced_specs = [
    VALIDATION_SPECS[0],   # difficulty 0
    VALIDATION_SPECS[3],   # difficulty 1 (first one after the 3 d0 entries)
    VALIDATION_SPECS[7],   # difficulty 2 (first one after the 3 d0 + 4 d1 entries)
]
traced_results = []
for spec in traced_specs:
    print(f"=== spec: {spec['path']} (difficulty {spec['difficulty']}) ===")
    result = run_traced_episode(spec, deterministic=True, verbose=True)
    traced_results.append(result)

## 16. Training curves and held-out test evaluation

Training curves show learning within the randomized curriculum. The final metrics are calculated on the disjoint test bank with deterministic action selection.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.ravel()
curves = [
    (reward_history, "mean rollout reward"),
    (loss_history, "mean PPO minibatch loss"),
    (episode_length_history, "mean completed-episode length"),
    (goal_success_rate_history, "training goal-success rate"),
    (coverage_history, "training host coverage"),
    (honeypot_rate_history, "training honeypot-touch rate"),
]
for axis, (values, title) in zip(axes, curves):
    axis.plot(values)
    axis.set_title(title)
    axis.set_xlabel("iteration")
for index in (3, 4, 5):
    axes[index].set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig("multiscenario_training_curves.png")
plt.show()

test_metrics = evaluate_scenario_specs(TEST_SPECS, deterministic=True)
print("Held-out test results")
print(f"  goal success:      {test_metrics['goal_success']:.2%}")
print(f"  host coverage:     {test_metrics['coverage']:.2%}")
print(f"  honeypot touched:  {test_metrics['honeypot_hit']:.2%}")
print(f"  mean reward:       {test_metrics['reward']:.2f}")
print(f"  mean episode steps:{test_metrics['length']:.1f}")

test_breakdown = evaluate_scenario_specs_by_difficulty(TEST_SPECS, deterministic=True)
print_difficulty_breakdown(test_breakdown, label="held-out test")

train_breakdown = evaluate_scenario_specs_by_difficulty(TRAIN_SPECS, deterministic=True)
print_difficulty_breakdown(train_breakdown, label="train (deterministic)")

if validation_history:
    validation_iterations = [item["iteration"] for item in validation_history]
    validation_success = [item["goal_success"] for item in validation_history]
    plt.plot(validation_iterations, validation_success, marker="o")
    plt.ylim(-0.02, 1.02)
    plt.xlabel("training iteration")
    plt.ylabel("held-out goal-success rate")
    plt.title("Validation generalization")
    plt.show()


## 17. Embedding visualization on an unseen test scenario

Runs the trained deterministic policy on a held-out network and colors graph embeddings by compromise coverage.


In [ ]:
embeddings_collected = []
labels_collected = []

viz_spec = TEST_SPECS[0]
env_viz, obs, info, viz_required_hosts = start_episode(viz_spec)

for step in range(env_viz.scenario.step_limit):
    with torch.no_grad():
        next_action, _, _, graph_emb, _, _ = select_policy_action(
            obs, env_viz, deterministic=True
        )
    next_obs, reward, terminated, truncated, info = step_env(
        env_viz, next_action
    )
    outcome = update_trackers(
        next_action, info, env_viz, edge_tracker,
        node_service_state, node_process_state, node_scan_state,
    )

    compromised = sum(
        env_viz.current_state.host_compromised(address)
        for address in viz_required_hosts
    )
    embeddings_collected.append(graph_emb.detach().numpy())
    labels_collected.append(compromised / len(viz_required_hosts))

    obs = next_obs
    if terminated or truncated:
        break

embeddings_arr = np.asarray(embeddings_collected)
if len(embeddings_arr) >= 2:
    reduced = PCA(n_components=2).fit_transform(embeddings_arr)
    plt.scatter(
        reduced[:, 0], reduced[:, 1],
        c=labels_collected, cmap="viridis", vmin=0, vmax=1,
    )
    plt.colorbar(label="non-honeypot host coverage")
    plt.title(f"Graph embeddings on unseen scenario: {viz_spec['name']}")
    plt.show()
else:
    print("Episode ended too quickly to plot PCA (need at least two steps).")
